In [ ]:
# Standard library
from pathlib import Path

# Third-party libraries
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import LinearSVC


In [3]:
"""
Configuration for training the TF-IDF + Linear SVM model.

This module defines:
- Project paths
- Dataset and model locations
- Hyperparameter grid for model selection
"""

# Data
try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    # fallback for notebooks or interactive environments
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR: Path = PROJECT_ROOT / "data" / "interim"
MODEL_DIR: Path = PROJECT_ROOT / "models" / "experiment"

DATA_FILE: str = "WELFake_Dataset.csv"
DATA_PATH: Path = DATA_DIR / DATA_FILE
MODEL_FILE: Path = MODEL_DIR / "best_tfidf_svm_model.pkl"

# Model hyperparameters
PARAM_GRID = [
    {   
        # Larger C => weaker regularization
        "C": [0.1, 1, 10],

        # Adjust weights to handle class imbalance
        "class_weight": [None, "balanced"],

        # Squared hinge is smoother and typically more stable
        "loss": ["squared_hinge"],

        # Dual formulation is recommended for LinearSVC
        "dual": [True],

        # Increase iterations to avoid convergence warnings
        "max_iter": [5000],
    }
]


In [4]:
def running_in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


def identity(x):
    return x

In [5]:
"""
 TF-IDF + Linear SVM Training

This notebook trains and evaluates a TF-IDF + Linear SVM classifier using
preprocessed train/test splits. Hyperparameters are optimized via GridSearchCV.
"""

X_train = joblib.load(DATA_DIR / "X_train.pkl")
y_train = joblib.load(DATA_DIR / "y_train.pkl")

X_test  = joblib.load(DATA_DIR / "X_test.pkl")
y_test  = joblib.load(DATA_DIR / "y_test.pkl")

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")

tfidf = TfidfVectorizer(
    
    tokenizer=identity,     
    preprocessor=identity,   
    token_pattern=None,

    # Skips ngrams occuring less the this value.
    # One value to reduce computation
    min_df=5,

    # Skips ngrams occuring more the this percentage.
    # One value to reduce computation
    max_df=0.95,

    # Testing only bigrams to save computation.
    ngram_range=(1,1)
)

tfidf.fit(X_train)  # freeze on full training set
X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

N_JOBS = 1 if running_in_notebook() else -1

grid = GridSearchCV(
    LinearSVC(),
    param_grid=PARAM_GRID,
    cv=5,
    scoring="accuracy",
    verbose=3,
    n_jobs=-1
)
grid.fit(X_train_tfidf, y_train)

print("Beste parameters:", grid.best_params_)
print("Beste CV-score:", grid.best_score_)

# Apply the best grid searched hyperparameters for the LSVC
y_pred = grid.predict(X_test_tfidf)

# Evaluate
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


# Save predictions
pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred
})

# Save trained model
joblib.dump(grid.best_estimator_, MODEL_FILE)

Train size: 57229
Test size:  14308
Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV 4/5] END C=0.1, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.961 total time=   2.1s
[CV 5/5] END C=0.1, class_weight=None, dual=True, loss=squared_hinge, max_iter=5000;, score=0.960 total time=   2.1s
[CV 3/5] END C=0.1, class_weight=None, dual=True, loss=squared_hinge, max_iter=5000;, score=0.961 total time=   2.1s
[CV 4/5] END C=0.1, class_weight=None, dual=True, loss=squared_hinge, max_iter=5000;, score=0.961 total time=   2.1s
[CV 3/5] END C=0.1, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.961 total time=   2.1s
[CV 1/5] END C=0.1, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.962 total time=   2.1s
[CV 2/5] END C=0.1, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.963 total time=   2.1s
[CV 2/5] END C=0.1, class_weight=None, dual=True, loss=squared_hinge,

['/Users/pietervanbrakel/zelfstudie/projects/scriptie/models/experiment/best_tfidf_svm_model.pkl']